# Silver Layer: Weather Creation

Inspects the approved Bronze JSON batch, checks array alignment, creates the Silver Weather table, and expands the raw JSON into one row per source-provided local-hour label.

**Prerequisite:** Run the Bronze setup, load, and validation notebooks first.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `ftw-week-08`.`02_silver`;


## Weather JSON Inspection

Inspect the actual JSON structure stored in the Bronze Weather table.

In [0]:
-- Check the Bronze Weather table structure and content
SELECT 
  source_system,
  source_url,
  source_file,
  batch_id,
  ingested_at,
  LENGTH(raw_json) AS json_length_bytes
FROM `ftw-week-08`.`01_bronze`.`weather_raw`

In [0]:
-- Inspect the JSON structure - look at the top-level keys and hourly data structure
WITH parsed AS (
  SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
)
SELECT 
  json_data.latitude AS latitude,
  json_data.longitude AS longitude,
  json_data.timezone AS timezone,
  json_data.timezone_abbreviation AS timezone_abbreviation,
  SIZE(json_data.hourly.time) AS hourly_time_count,
  SIZE(json_data.hourly.temperature_2m) AS hourly_temperature_count,
  SIZE(json_data.hourly.precipitation) AS hourly_precipitation_count,
  SIZE(json_data.hourly.rain) AS hourly_rain_count,
  SIZE(json_data.hourly.snowfall) AS hourly_snowfall_count,
  SIZE(json_data.hourly.weather_code) AS hourly_weather_code_count,
  SIZE(json_data.hourly.wind_speed_10m) AS hourly_wind_speed_count
FROM parsed

In [0]:
-- Sample first and last few hourly timestamps from the JSON payload
WITH parsed AS (
  SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
)
SELECT 
  json_data.hourly.time[0] AS first_timestamp,
  json_data.hourly.time[1] AS second_timestamp,
  json_data.hourly.time[2] AS third_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 3] AS third_to_last_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 2] AS second_to_last_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 1] AS last_timestamp
FROM parsed

## Weather Silver Transformation

Create the Silver Weather table by parsing and flattening the JSON hourly arrays into one row per hourly timestamp.

In [0]:
-- Validate that all hourly arrays have the same length before POSEXPLODE
WITH parsed AS (
  SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
),
array_sizes AS (
  SELECT
    CASE WHEN json_data IS NOT NULL THEN 'SUCCESS' ELSE 'FAILED' END AS json_parse_status,
    SIZE(json_data.hourly.time) AS time_count,
    SIZE(json_data.hourly.temperature_2m) AS temperature_2m_count,
    SIZE(json_data.hourly.precipitation) AS precipitation_count,
    SIZE(json_data.hourly.rain) AS rain_count,
    SIZE(json_data.hourly.snowfall) AS snowfall_count,
    SIZE(json_data.hourly.weather_code) AS weather_code_count,
    SIZE(json_data.hourly.wind_speed_10m) AS wind_speed_10m_count
  FROM parsed
)
SELECT
  json_parse_status,
  time_count,
  temperature_2m_count,
  precipitation_count,
  rain_count,
  snowfall_count,
  weather_code_count,
  wind_speed_10m_count,
  CASE
    WHEN json_parse_status = 'FAILED' THEN 'FAIL'
    WHEN time_count = 0 THEN 'FAIL'
    WHEN time_count = temperature_2m_count
     AND time_count = precipitation_count
     AND time_count = rain_count
     AND time_count = snowfall_count
     AND time_count = weather_code_count
     AND time_count = wind_speed_10m_count
    THEN 'PASS'
    ELSE 'FAIL'
  END AS array_alignment_status
FROM array_sizes

In [0]:
%sql
-- Schema-only creation: run once to initialize table if it does not exist
CREATE TABLE IF NOT EXISTS `ftw-week-08`.`02_silver`.`weather` (
  weather_datetime TIMESTAMP_NTZ, 
  latitude DOUBLE, 
  longitude DOUBLE,
  timezone STRING, 
  timezone_abbreviation STRING, 
  utc_offset_seconds INT,
  temperature_2m DOUBLE, 
  precipitation DOUBLE, 
  rain DOUBLE,
  snowfall DOUBLE, 
  weather_code INT, 
  wind_speed_10m DOUBLE,
  source_system STRING, 
  source_url STRING, 
  source_file STRING,
  batch_id STRING, 
  ingested_at TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported');

In [0]:
%sql
-- Incremental execution: run on every pipeline execution
-- Appends only new JSON batches based on batch_id
INSERT INTO `ftw-week-08`.`02_silver`.`weather`
WITH parsed_json AS (
  SELECT
    from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING,
      timezone_abbreviation STRING, utc_offset_seconds INT,
      hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>,
        precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>,
        snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>,
        wind_speed_10m: ARRAY<DOUBLE>>') AS json_data,
    source_system, source_url, source_file, batch_id, ingested_at
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE batch_id NOT IN (
    SELECT DISTINCT batch_id FROM `ftw-week-08`.`02_silver`.`weather`
  )
),
exploded_weather AS (
  SELECT POSEXPLODE(json_data.hourly.time) AS (hour_index, hourly_time_str),
         json_data, source_system, source_url, source_file, batch_id, ingested_at
  FROM parsed_json
)
SELECT
  TRY_CAST(hourly_time_str AS TIMESTAMP_NTZ) AS weather_datetime,
  json_data.latitude, json_data.longitude, json_data.timezone,
  json_data.timezone_abbreviation, json_data.utc_offset_seconds,
  json_data.hourly.temperature_2m[hour_index] AS temperature_2m,
  json_data.hourly.precipitation[hour_index]  AS precipitation,
  json_data.hourly.rain[hour_index]            AS rain,
  json_data.hourly.snowfall[hour_index]        AS snowfall,
  json_data.hourly.weather_code[hour_index]    AS weather_code,
  json_data.hourly.wind_speed_10m[hour_index]  AS wind_speed_10m,
  source_system, source_url, source_file, batch_id, ingested_at
FROM exploded_weather
ORDER BY weather_datetime;

In [ ]:
%sql
-- Built-in verification: confirm total weather records and latest weather timestamp after load
SELECT COUNT(*) AS total_weather_hours, MAX(weather_datetime) AS latest_hour
FROM `ftw-week-08`.`02_silver`.`weather`;

In [ ]:
%sql
-- Databricks notebook source
-- Modular source: Silver Weather table (incremental)

-- Schema-only creation: run once to initialize table if it does not exist
-- TIMESTAMP_NTZ preserves local-hour labels without session timezone conversion
CREATE TABLE IF NOT EXISTS `ftw-week-08`.`02_silver`.`weather` (
  weather_datetime TIMESTAMP_NTZ,
  latitude DOUBLE,
  longitude DOUBLE,
  timezone STRING,
  timezone_abbreviation STRING,
  utc_offset_seconds INT,
  temperature_2m DOUBLE,
  precipitation DOUBLE,
  rain DOUBLE,
  snowfall DOUBLE,
  weather_code INT,
  wind_speed_10m DOUBLE,
  source_system STRING,
  source_url STRING,
  source_file STRING,
  batch_id STRING,
  ingested_at TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported');

-- COMMAND ----------

-- Incremental execution: appends only new JSON batches not already present in Silver
INSERT INTO `ftw-week-08`.`02_silver`.`weather`
WITH parsed_json AS (
  SELECT
    from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING,
      timezone_abbreviation STRING, utc_offset_seconds INT,
      hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>,
        precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>,
        snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>,
        wind_speed_10m: ARRAY<DOUBLE>>') AS json_data,
    source_system,
    source_url,
    source_file,
    batch_id,
    ingested_at
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE batch_id NOT IN (
    SELECT DISTINCT batch_id FROM `ftw-week-08`.`02_silver`.`weather`
  )
),
exploded_weather AS (
  SELECT
    POSEXPLODE(json_data.hourly.time) AS (hour_index, hourly_time_str),
    json_data,
    source_system,
    source_url,
    source_file,
    batch_id,
    ingested_at
  FROM parsed_json
)
SELECT
  TRY_CAST(hourly_time_str AS TIMESTAMP_NTZ) AS weather_datetime,
  json_data.latitude AS latitude,
  json_data.longitude AS longitude,
  json_data.timezone AS timezone,
  json_data.timezone_abbreviation AS timezone_abbreviation,
  json_data.utc_offset_seconds AS utc_offset_seconds,
  json_data.hourly.temperature_2m[hour_index] AS temperature_2m,
  json_data.hourly.precipitation[hour_index] AS precipitation,
  json_data.hourly.rain[hour_index] AS rain,
  json_data.hourly.snowfall[hour_index] AS snowfall,
  json_data.hourly.weather_code[hour_index] AS weather_code,
  json_data.hourly.wind_speed_10m[hour_index] AS wind_speed_10m,
  source_system,
  source_url,
  source_file,
  batch_id,
  ingested_at
FROM exploded_weather
ORDER BY weather_datetime;
